In [ ]:
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
import os
import pandas as pd
import random

In [ ]:
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7,static_image_mode=True, min_tracking_confidence=0.7)
mp_drawing = mp.solutions.drawing_utils

np.random.seed(42)
random.seed(42)

num_classes = 26

def get_data(folder, img_per_class, img_total_per_class, ext):
    img_nb = num_classes * img_per_class
    X = np.empty((img_nb, 63), dtype=np.float32)
    y = np.empty((img_nb,), dtype=int)

    label_map = {chr(65+i): i for i in range(26)}
    cnt = 0
    
    for letter in label_map.keys():
        print(f"Traitement lettre {letter}...")
        folder_path = os.path.join(folder, letter)
        cnt_letter = 0
        img_indice = np.random.permutation(np.arange(1, img_total_per_class + 1))
        for img_num in img_indice:
            img_path = os.path.join(folder_path, letter+str(img_num)+ext)
            img_file = cv2.imread(img_path)
            image_rgb = cv2.cvtColor(img_file, cv2.COLOR_BGR2RGB)
            results = hands.process(image_rgb)
            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    points_coord = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark])
                    X[cnt] = points_coord.reshape((63,))
                    y[cnt] = label_map[letter]
                    cnt += 1
                    cnt_letter +=1
                    break
                    
            if cnt_letter >= img_per_class:
                break
                
    return X[:cnt], y[:cnt]

In [ ]:
img_per_class_1 = 400
img_total_per_class_1 = 3000

train_dir = '../data/images/one_hand'

X_1, y_1 = get_data(train_dir, img_per_class_1, img_total_per_class_1,".jpg")

df_1 = pd.DataFrame(X_1, columns=[f"p{i}" for i in range(63)])
df_1["label"] = y_1.astype(int)
df_1.to_csv("../data/asl_one_hand.csv", index=False)

In [ ]:
img_per_class_2 = 100
img_total_per_class_2 = 100

test_dir = '../data/images/two_hands'

X_2, y_2 = get_data(test_dir, img_per_class_2, img_total_per_class_2, ".png")
df_2 = pd.DataFrame(X_2, columns=[f"p{i}" for i in range(63)])
df_2["label"] = y_2.astype(int)
df_2.to_csv("../data/asl_two_hands.csv", index=False)